# Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [16]:
import warnings
warnings.filterwarnings('ignore')

In [17]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

This notebook uses **Anthropic Claude Haiku** (`ChatAnthropic`), same model id as `prompts/langchain_models_prompts_output_parsers.ipynb` and `memory/langchain_memory.ipynb`. Set `ANTHROPIC_API_KEY` in your `.env`.

> **If you see `NotFoundError` / 404 for `model:`:** the model ID is wrong or retired. Update `llm_model` in the next code cell to a current Haiku id from [Anthropic’s models page](https://docs.anthropic.com/en/docs/about-claude/models).

The course video uses OpenAI; chain concepts are unchanged.

In [27]:
# Claude Haiku — same default as other notebooks in this repo
llm_model = "claude-haiku-4-5-20251001"

In [19]:
#!pip install pandas

In [20]:
import pandas as pd
df = pd.read_csv('Data.csv')

In [21]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [26]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [28]:
llm = ChatAnthropic(temperature=0.9, model=llm_model, max_tokens=1024)

In [30]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [31]:
chain = LLMChain(llm=llm, prompt=prompt)

In [32]:
product = "Queen Size Sheet Set"
print(chain.run(product))

'# Best Names for a Queen Size Sheet Set Company\n\nHere are some strong options:\n\n## Direct & Clear\n- **Royal Bedding** - suggests quality and the "royal" size\n- **Queen Linens** - straightforward and professional\n- **Premium Sheet Co.** - emphasizes quality\n\n## Lifestyle-Focused\n- **Luxe Sheets** - appeals to comfort seekers\n- **Cozy Haven** - warm, inviting feeling\n- **Silk & Serenity** - suggests comfort and quality materials\n\n## Modern/Trendy\n- **Nest Bedding** - contemporary, comfort-focused\n- **Threshold Home** - modern, accessible luxury\n- **Draper + Co.** - upscale, artisanal feel\n\n## My Top Recommendation\n**Royal Linens** or **The Linen Company** — because they:\n- Clearly indicate the product\n- Sound established and trustworthy\n- Work well across marketing channels\n- Are easy to remember and spell\n\nThe best choice depends on your brand positioning:\n- **Luxury market?** → Luxe, Silk & Serenity\n- **Practical/everyday?** → Royal Linens, Queen Linens\n- 

## SimpleSequentialChain

In [33]:
from langchain_classic.chains import SimpleSequentialChain

In [34]:
llm = ChatAnthropic(temperature=0.9, model=llm_model, max_tokens=1024)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [35]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [37]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [38]:
print(overall_simple_chain.run(product))



> Entering new SimpleSequentialChain chain...
# Best Names for a Queen Size Sheet Set Company

Here are some strong options:

## Direct/Descriptive
- **Royal Linens** - conveys luxury and the "royal" connection to "queen"
- **Queen's Rest** - simple, memorable, and product-focused
- **Premium Bedding Co.** - straightforward and professional

## Creative/Playful
- **The Linen Crown** - clever wordplay on royalty
- **Majesty Sheets** - elegant and regal
- **Regal Comfort** - combines quality with luxury positioning

## Modern/Lifestyle
- **Luxe Bedding** - contemporary and upscale
- **Comfort Collective** - trendy, inclusive feel
- **Thread & Crown** - sophisticated and unique

## My Top Recommendation
**Royal Linens** or **The Linen Crown** work best because they:
- Subtly reference "queen" without being too literal
- Sound premium and trustworthy
- Are memorable and distinctive
- Work well for a brand that might expand beyond queen sizes

The best choice depends on your target market

## SequentialChain

In [42]:
from langchain_classic.chains import SequentialChain

In [43]:
llm = ChatAnthropic(temperature=0.9, model=llm_model, max_tokens=1024)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )


In [44]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )


In [45]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [46]:

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [47]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)

In [48]:
from pprint import pprint

review = df.Review[5]
pprint(overall_chain(review))



> Entering new SequentialChain chain...

> Finished chain.
{'English_Review': '# English Translation\n'
                   '\n'
                   "I find the taste mediocre. The foam doesn't hold up, it's "
                   'odd. I buy the same ones from stores and the taste is much '
                   'better...\n'
                   'Old batch or counterfeit!?',
 'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. "
           "J'achète les mêmes dans le commerce et le goût est bien "
           'meilleur...\n'
           'Vieux lot ou contrefaçon !?',
 'followup_message': '# Réponse de suivi\n'
                     '\n'
                     'Merci beaucoup pour votre avis détaillé. Nous sommes '
                     "vraiment désolés d'apprendre que le produit n'a pas "
                     'répondu à vos attentes, particulièrement concernant le '
                     'goût et la qualité de la mousse.\n'
                     '\n'
                     

## Router Chain

In [49]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [50]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [52]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain_core.prompts import PromptTemplate

In [53]:
llm = ChatAnthropic(temperature=0, model=llm_model, max_tokens=1024)

In [54]:

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [64]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [57]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [58]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [60]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [65]:
print(chain.run("What is black body radiation?"))



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.
# Black Body Radiation

Great question! Here's the essence:

## The Basic Idea
A **black body** is an idealized object that absorbs all light and radiation that hits it (reflecting none). When heated, it re-emits that energy as electromagnetic radiation.

## Key Points

**What it radiates:**
- The hotter the object, the more radiation it emits
- The hotter it gets, the radiation shifts to shorter wavelengths (higher frequencies)
  - Cool objects → infrared radiation (heat we feel)
  - Very hot objects → visible light (like a glowing metal)
  - Extremely hot objects → ultraviolet and beyond

**Why it matters:**
- It's a fundamental concept in physics—real objects approximate black body behavior
- The Sun, Earth, and even you emit black body radiation
- It helped launch quantum mechanics! Classical physics couldn't explain the observed spectrum, but Planck's quantum theory sol

In [67]:
print(chain.run("what is 2 + 2"))



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.
# Breaking Down 2 + 2

Let me work through this step-by-step:

## Component Parts
- **First number:** 2
- **Operation:** Addition (+)
- **Second number:** 2

## Solving Each Part
When we add 2 and 2, we're combining two groups of 2 items each.

## Putting It Together
2 + 2 = **4**

---

**Explanation:** If you have 2 apples and someone gives you 2 more apples, you now have a total of 4 apples.


In [66]:
print(chain.run("Why does every cell in our body contain DNA?"))



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}

ValueError: Received invalid destination chain name 'biology'

Reminder: Download your notebook to you local computer to save your work.